# ingest.ipynb
**Step 1 of 3 — Data Acquisition & Cleaning**

Downloads the ULB Credit Card Fraud Detection dataset from Kaggle, validates the expected schema, cleans the data, and outputs `cleaned.csv`.

**Run order:** `ingest.ipynb` → `enrich.ipynb` → `load_mongo.ipynb`

## 1. Install Dependencies

In [ ]:
# Install the Kaggle CLI if not already available
!pip install kaggle --quiet

## 2. Imports & Logging Setup

In [ ]:
import os
import logging
import pandas as pd

# Configure logging to both file and notebook output
logging.basicConfig(
    filename="ingest.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

# Also print logs to the notebook
console = logging.StreamHandler()
console.setLevel(logging.INFO)
logging.getLogger("").addHandler(console)

print("Imports complete.")

## 3. Configuration

In [ ]:
# Expected columns in the raw ULB dataset
REQUIRED_COLUMNS = (
    ["Time"] +
    [f"V{i}" for i in range(1, 29)] +
    ["Amount", "Class"]
)

RAW_PATH = "creditcard.csv"
OUTPUT_PATH = "cleaned.csv"

print(f"Expecting {len(REQUIRED_COLUMNS)} columns.")
print(f"Output will be saved to: {OUTPUT_PATH}")

## 4. Download Dataset from Kaggle

> **Setup:** Place your `kaggle.json` API key in `~/.kaggle/kaggle.json` (or `/root/.kaggle/kaggle.json` in Colab) before running this cell.
> You can download your key from https://www.kaggle.com/settings → API → Create New Token.

In [ ]:
def download_dataset(raw_path: str) -> None:
    """Download the ULB fraud dataset from Kaggle using the Kaggle CLI."""
    if os.path.exists(raw_path):
        logging.info(f"{raw_path} already exists, skipping download.")
        print(f"{raw_path} already exists — skipping download.")
        return
    try:
        logging.info("Downloading dataset from Kaggle...")
        os.system("kaggle datasets download -d mlg-ulb/creditcardfraud --unzip")
        logging.info("Download complete.")
        print("Download complete.")
    except Exception as e:
        logging.error(f"Download failed: {e}")
        raise

download_dataset(RAW_PATH)

## 5. Load Raw Data

In [ ]:
def load_raw(path: str) -> pd.DataFrame:
    """Load the raw CSV into a dataframe."""
    logging.info(f"Loading raw data from {path}")
    try:
        df = pd.read_csv(path)
        logging.info(f"Loaded {len(df)} rows and {len(df.columns)} columns.")
        return df
    except FileNotFoundError as e:
        logging.error(f"File not found: {e}")
        raise

df_raw = load_raw(RAW_PATH)
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 6. Validate Schema

In [ ]:
def validate_schema(df: pd.DataFrame, required_columns: list) -> None:
    """Validate that all expected columns are present in the dataframe."""
    logging.info("Validating schema...")
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        logging.error(f"Missing columns: {missing}")
        raise ValueError(f"Schema validation failed. Missing columns: {missing}")
    logging.info("Schema validation passed.")
    print(f"Schema valid. All {len(required_columns)} expected columns present.")

validate_schema(df_raw, REQUIRED_COLUMNS)

## 7. Clean Data

In [ ]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the dataframe:
    - Drop duplicate rows
    - Drop rows with any null values
    - Ensure Class is integer (0 or 1)
    - Ensure Amount is non-negative
    """
    logging.info(f"Starting cleaning. Rows before: {len(df)}")

    df = df.drop_duplicates()
    logging.info(f"After dropping duplicates: {len(df)} rows")

    df = df.dropna()
    logging.info(f"After dropping nulls: {len(df)} rows")

    # Validate Class values
    invalid_class = df[~df["Class"].isin([0, 1])]
    if not invalid_class.empty:
        logging.warning(f"Dropping {len(invalid_class)} rows with invalid Class values.")
        df = df[df["Class"].isin([0, 1])]

    df["Class"] = df["Class"].astype(int)

    # Validate Amount
    invalid_amount = df[df["Amount"] < 0]
    if not invalid_amount.empty:
        logging.warning(f"Dropping {len(invalid_amount)} rows with negative Amount.")
        df = df[df["Amount"] >= 0]

    logging.info(f"Cleaning complete. Rows after: {len(df)}")
    logging.info(f"Fraud rate: {df['Class'].mean():.4%}")
    return df

df_clean = clean(df_raw)
print(f"Final shape: {df_clean.shape}")
print(f"Fraud rate: {df_clean['Class'].mean():.4%}")
df_clean.head()

## 8. Summary Statistics

In [ ]:
print("=== Class Distribution ===")
print(df_clean["Class"].value_counts())
print()
print("=== Amount Statistics ===")
print(df_clean["Amount"].describe().round(2))

## 9. Save Cleaned Data

In [ ]:
df_clean.to_csv(OUTPUT_PATH, index=False)
logging.info(f"Cleaned data saved to {OUTPUT_PATH}")
print(f"Done. {len(df_clean)} rows saved to '{OUTPUT_PATH}'.")
print("Next step: run enrich.ipynb")